In [1]:
import calendar
import pandas as pd
from datetime import date, timedelta
from sqlalchemy import create_engine
from pandas.tseries.offsets import BDay

engine = create_engine("sqlite:///c:\\ruby\\port_lite\\db\\development.sqlite3")
conlite = engine.connect()

engine = create_engine("mysql+pymysql://root:@localhost:3306/stock")
const = engine.connect()

pd.set_option('display.max_row',None)

data_path = "../data/"
csv_path = "\\Users\\User\\iCloudDrive\\"
box_path = "\\Users\\User\\Dropbox\\"
one_path = "\\Users\\User\\OneDrive\\Documents\\Data\\"

today = date.today()
yesterday = today - timedelta(days=1)
today, yesterday

(datetime.date(2023, 1, 14), datetime.date(2023, 1, 13))

In [2]:
# specify the number of business days
num_days = 1
# convert the timedelta object to a BusinessDay object
num_business_days = BDay(num_days)
yesterday = today - num_business_days
#yesterday = yesterday.date()
print(f'today: {today}')
print(f'yesterday: {yesterday}')

today: 2023-01-14
yesterday: 2023-01-13 00:00:00


In [3]:
yesterday = yesterday.date()
a_year_ago = yesterday - timedelta(days=365)
yesterday, a_year_ago

(datetime.date(2023, 1, 13), datetime.date(2022, 1, 13))

### Get past one quarter data

In [4]:
sql = """
SELECT name
FROM buy 
ORDER BY name
"""
df = pd.read_sql(sql, const)

names = df['name'].values.tolist()
in_p = ", ".join(map(lambda name: "'%s'" % name, names))
in_p

"'ASP', 'BANPU', 'BCH', 'CPNREIT', 'DCC', 'DIF', 'GVREIT', 'IVL', 'JASIF', 'KCE', 'MAKRO', 'MCS', 'NER', 'ORI', 'PTTGC', 'RCL', 'SCC', 'SCCC', 'SENA', 'STA', 'SYNEX', 'TFFIF', 'TMT', 'WHAIR', 'WHART'"

In [5]:
in_p = "'JMART', 'SINGER'"
#in_p = "'TTLPF'"
in_p

"'JMART', 'SINGER'"

In [6]:
#one_qtr_ago = yesterday - timedelta(days=4)
one_qtr_ago = date(2023, 1, 9)

sql = """
SELECT name, date, price, qty, maxp, minp
FROM price 
WHERE date >= '%s' AND name IN (%s) 
ORDER BY name, date"""
sql = sql % (one_qtr_ago, in_p)
print(sql)


SELECT name, date, price, qty, maxp, minp
FROM price 
WHERE date >= '2023-01-09' AND name IN ('JMART', 'SINGER') 
ORDER BY name, date


In [7]:
df = pd.read_sql(sql, const)
df

,name,date,price,qty,maxp,minp
0,JMART,2023-01-09,41.50,10986578,42.00,40.25
1,JMART,2023-01-10,42.00,9751872,42.75,41.50
2,JMART,2023-01-11,42.25,5769834,42.50,41.75
3,JMART,2023-01-12,42.50,4643436,42.75,41.75
4,JMART,2023-01-13,42.50,10655144,43.50,42.00
5,SINGER,2023-01-09,29.75,15522537,30.00,27.00
6,SINGER,2023-01-10,29.25,4615984,30.00,29.00
7,SINGER,2023-01-11,29.50,2719692,29.75,29.25
8,SINGER,2023-01-12,30.00,6117144,30.50,29.50
9,SINGER,2023-01-13,30.25,8847453,31.25,30.00


In [8]:
data_path = "../data/"
file_name = "Quarterly-Price-by-Name.csv"
output_file = data_path + file_name

df.set_index("name", inplace=True)
df.to_csv(output_file, header=None)